<a href="https://colab.research.google.com/github/worasak2530seed/fast-hybrid-pro/blob/main/notebooks/Early_Signal_Scanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fast Hybrid Pro — EARLY ENTRY / EARLY WARNING Scanner

**Standalone live-data scanner. This is not the Production notebook.**

- Reads current SET/mai classification from SET's public listing endpoint, with a public SET-sourced mirror fallback.
- Uses Yahoo Finance (`yfinance`) for OHLCV and current market-cap lookup; no SETSMART or SMART Marketplace.
- Applies the agreed universe filters before downloading the longer indicator history.
- Runs only the latest completed daily bar. It does **not** run a backtest, parameter sweep, place orders, write to `main`, or trigger GitHub Actions.
- The 20-day trading value is estimated as `Close × Volume` because Yahoo OHLCV does not provide the exchange's exact daily traded-value field in this workflow. Review this limitation before relying on borderline cases.

Run the cells from top to bottom. The first scan may take several minutes due to Yahoo Finance requests.

In [6]:
# Install runtime dependencies and check out the isolated scanner branch.
%pip -q install yfinance pandas numpy requests openpyxl xlrd

import io
import re
import sys
import time
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from IPython.display import display

REPO_URL = "https://github.com/worasak2530seed/fast-hybrid-pro.git"
BRANCH = "main"
REPO_DIR = Path("/content/fast-hybrid-pro-scanner-source")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from src.early_signal_scanner import CONFIG, scan_universe

print("SCANNER_BRANCH_CHECKOUT_PASS")
print("CONFIG:", CONFIG)


SCANNER_BRANCH_CHECKOUT_PASS
CONFIG: ScannerConfig(macd_fast=4, macd_slow=12, macd_signal=9, stoch_window=8, stoch_smooth_k=3, stoch_smooth_d=3, ema_fast=5, ema_mid=10, ema_long=75, bb_window=20, bb_std=2.0, kc_atr_window=14, kc_multiplier=1.5, squeeze_lookback=10, minimum_score=3, target1_pct=0.07, target2_pct=0.15, max_holding_days_reference=20)


In [ ]:
# Current market-data scan — no backtest or order execution.

UNIVERSE_MARKETS = {"SET", "MAI"}
EXCLUDED_SECTORS = {"BANK", "FIN", "INSUR", "PF&REIT", "PROP", "TOURISM"}
MIN_MARKET_CAP_MBAHT = 5000.0
MIN_TOTAL20_VALUE_MBAHT = 20.0
MIN_PRICE_BAHT = 1.0
MAX_PRICE_BAHT = 80.0
CLASSIFICATION_SOURCE_USED = None
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; Fast-Hybrid-Pro-Scanner/1.0)", "Accept": "*/*"}

def _flatten_columns(frame):
    frame = frame.copy()
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = [" ".join(str(x).strip() for x in col if str(x).strip() and str(x) != "nan") for col in frame.columns]
    else:
        frame.columns = [str(c).strip() for c in frame.columns]
    return frame

def _find_col(frame, names):
    lookup = {str(c).strip().lower(): c for c in frame.columns}
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    return None

def _normalize_classification(raw):
    raw = _flatten_columns(raw)
    symbol_col = _find_col(raw, ["symbol", "ticker"])
    market_col = _find_col(raw, ["market", "market segment"])
    sector_col = _find_col(raw, ["sector", "sector name"])
    company_col = _find_col(raw, ["company name", "company", "name", "security name"])
    industry_col = _find_col(raw, ["industry group", "industry"])
    if not all([symbol_col, market_col, sector_col]):
        raise ValueError(f"Classification missing required fields; columns={list(raw.columns)}")
    out = pd.DataFrame({
        "Symbol": raw[symbol_col].astype(str).str.strip().str.upper(),
        "Market": raw[market_col].astype(str).str.strip().str.upper(),
        "Sector": raw[sector_col].astype(str).str.strip().str.upper(),
        "Industry": raw[industry_col].astype(str).str.strip().str.upper() if industry_col else "",
        "Company": raw[company_col].astype(str).str.strip() if company_col else "",
    })
    if not company_col:
        raise ValueError("Classification has no company-name field; cannot safely enforce ordinary-share-only gate")
    ordinary = out["Company"].str.contains(r"PUBLIC COMPANY LIMITED|PUBLIC CO\.?\s*,?\s*LTD\.?|PCL\.?$", case=False, regex=True, na=False)
    out = out[ordinary].copy()
    sector_map = {
        "BANKING": "BANK", "FINANCE & SECURITIES": "FIN", "INSURANCE": "INSUR",
        "PROPERTY DEVELOPMENT": "PROP", "PROPERTY FUND & REITS": "PF&REIT",
        "TOURISM & LEISURE": "TOURISM",
    }
    out["Sector"] = out["Sector"].replace(sector_map)
    out = out[out["Market"].isin(UNIVERSE_MARKETS)]
    out = out[out["Symbol"].ne("") & out["Symbol"].ne("NAN")].drop_duplicates("Symbol")
    if out.empty:
        raise ValueError("No SET/mai listed companies remained after classification normalization")
    return out.reset_index(drop=True)

def load_set_classification():
    global CLASSIFICATION_SOURCE_USED
    official_url = "https://www.set.or.th/dat/eod/listedcompany/static/listedCompanies_en_US.xls"
    try:
        response = requests.get(official_url, headers=HEADERS, timeout=30)
        response.raise_for_status()
        raw = None
        try:
            raw = pd.read_excel(io.BytesIO(response.content))
        except Exception:
            tables = pd.read_html(io.BytesIO(response.content))
            if tables:
                raw = tables[0]
        if raw is not None:
            result = _normalize_classification(raw)
            CLASSIFICATION_SOURCE_USED = official_url
            return result
    except Exception as exc:
        print("Official SET listing unavailable; trying public mirror:", type(exc).__name__)
    mirror_url = "https://raw.githubusercontent.com/lumduan/thai-securities-data/main/thai_securities_all_en.json"
    response = requests.get(mirror_url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    payload = response.json()
    raw = pd.DataFrame(payload if isinstance(payload, list) else list(payload.values()))
    result = _normalize_classification(raw)
    CLASSIFICATION_SOURCE_USED = mirror_url
    return result

def normalize_ohlcv(raw):
    if raw is None or raw.empty:
        return pd.DataFrame()
    frame = raw.copy()
    if isinstance(frame.columns, pd.MultiIndex):
        levels = [set(map(str, frame.columns.get_level_values(i))) for i in range(frame.columns.nlevels)]
        price_names = {"Open", "High", "Low", "Close", "Volume"}
        chosen = next((i for i, values in enumerate(levels) if price_names.issubset(values)), 0)
        frame.columns = frame.columns.get_level_values(chosen)
    frame = frame.loc[:, ~frame.columns.duplicated()].copy()
    rename = {str(c).strip().title(): str(c).strip().title() for c in frame.columns}
    frame.columns = [rename[str(c).strip().title()] for c in frame.columns]
    required = ["Open", "High", "Low", "Close", "Volume"]
    missing = [c for c in required if c not in frame.columns]
    if missing:
        raise ValueError(f"Missing OHLCV columns {missing}; got {list(frame.columns)}")
    frame = frame[required].copy()
    frame.index = pd.to_datetime(frame.index)
    if getattr(frame.index, "tz", None) is not None:
        frame.index = frame.index.tz_localize(None)
    for col in required:
        frame[col] = pd.to_numeric(frame[col], errors="coerce")
    frame = frame.dropna(subset=["Open", "High", "Low", "Close", "Volume"])
    frame = frame[~frame.index.duplicated(keep="last")].sort_index()
    # Exclude today's possibly incomplete daily candle (Thailand local date).
    today_bkk = pd.Timestamp.now(tz="Asia/Bangkok").date()
    frame = frame[frame.index.date < today_bkk]
    return frame

def get_market_cap_baht(ticker):
    try:
        value = yf.Ticker(ticker).fast_info.get("market_cap")
        if value is not None and np.isfinite(float(value)):
            return float(value)
    except Exception:
        pass
    try:
        value = yf.Ticker(ticker).info.get("marketCap")
        if value is not None and np.isfinite(float(value)):
            return float(value)
    except Exception:
        pass
    return np.nan

def extract_ticker_frame(batch_frame, ticker, batch_size):
    if batch_frame is None or batch_frame.empty:
        return pd.DataFrame()
    if isinstance(batch_frame.columns, pd.MultiIndex):
        for level in range(batch_frame.columns.nlevels):
            if ticker in set(map(str, batch_frame.columns.get_level_values(level))):
                try:
                    item = batch_frame.xs(ticker, axis=1, level=level, drop_level=True)
                    return normalize_ohlcv(item)
                except Exception:
                    continue
        if batch_size == 1:
            return normalize_ohlcv(batch_frame)
        return pd.DataFrame()
    return normalize_ohlcv(batch_frame) if batch_size == 1 else pd.DataFrame()

def download_yahoo_history(ticker, period, attempts=2):
    """Download one ticker with bounded retries; empty symbol results fail quickly."""
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            raw = yf.download(
                ticker, period=period, interval="1d", auto_adjust=False,
                progress=False, threads=False, timeout=20,
            )
            frame = normalize_ohlcv(raw)
            if not frame.empty:
                return frame
            last_error = "Yahoo returned no OHLCV rows"
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
        if attempt < attempts:
            time.sleep(1.5 * attempt)
    print(f"YAHOO_TICKER_UNAVAILABLE {ticker} | {last_error}")
    return pd.DataFrame()

def get_official_set_latest_trading_date():
    """
    Read a recent trading date from official SET pages.
    Ignore historical dates embedded in stale/static HTML and fail closed
    unless a recent date is found on an official page.
    """
    from html import unescape
    from datetime import timedelta

    reference_urls = [
        "https://www.set.or.th/en/market/statistics/five-days",
        "https://www.set.or.th/en/market/product/stock/overview?market=SET",
    ]
    date_patterns = [
        re.compile(r"\b\d{1,2}\s+[A-Za-z]{3}\s+\d{4}\b"),
        re.compile(r"\b\d{4}-\d{2}-\d{2}\b"),
    ]
    today = pd.Timestamp.now(tz="Asia/Bangkok").date()
    earliest_acceptable = today - timedelta(days=14)
    date_candidates = []
    pages_read = []

    for reference_url in reference_urls:
        try:
            response = requests.get(reference_url, headers=HEADERS, timeout=30)
            response.raise_for_status()
        except Exception as exc:
            print(f"SET date source unavailable: {reference_url} ({type(exc).__name__})")
            continue

        pages_read.append(reference_url)
        candidate_text = [response.text]
        try:
            tables = pd.read_html(io.StringIO(response.text))
        except (ValueError, ImportError):
            tables = []
        for table in tables:
            candidate_text.extend(table.columns.astype(str).tolist())
            candidate_text.extend(table.astype(str).to_numpy().ravel().tolist())

        for raw_text in candidate_text:
            text_value = unescape(re.sub(r"<[^>]+>", " ", str(raw_text)))
            found = []
            for pattern in date_patterns:
                found.extend(pattern.findall(text_value))
            for match in found:
                parsed = pd.to_datetime(match, errors="coerce", dayfirst=True)
                if pd.isna(parsed):
                    continue
                day = parsed.date()
                # Static HTML can contain historical dates (e.g. 2022).
                # Only trust recent dates; the report still fails closed if
                # no current market date can be verified.
                # The scanner intentionally excludes today's possibly incomplete
                # Yahoo candle, so today's calendar date (including a page's
                # generic "Last Update" date on a SET holiday) is not a valid
                # completed trading-session reference.
                if earliest_acceptable <= day < today:
                    date_candidates.append((day, reference_url))

    if not date_candidates:
        raise RuntimeError(
            "OFFICIAL_MARKET_DATE_UNAVAILABLE: no recent date (within 14 days) "
            "could be parsed from official SET pages. No report will be published."
        )

    latest_date, source_url = max(date_candidates, key=lambda item: item[0])
    print("Official SET market-date reference:", source_url)
    print("Official SET pages checked:", len(pages_read))
    print("Official SET latest published trading date:", latest_date)
    return latest_date

print("Loading SET/mai listing and sector classification...")
listed = load_set_classification()
listed = listed[~listed["Sector"].isin(EXCLUDED_SECTORS)].copy()
print("Classification source:", CLASSIFICATION_SOURCE_USED)
print("Candidates after sector exclusion:", len(listed))

# Stage 1: download short OHLCV only to apply price and 20-session liquidity gates.
symbols = listed["Symbol"].tolist()
tickers = [f"{symbol}.BK" for symbol in symbols]
short_data = {}
missing_short = []
BATCH_SIZE = 20
for start in range(0, len(tickers), BATCH_SIZE):
    batch = tickers[start:start + BATCH_SIZE]
    print(f"Downloading universe snapshot {start + 1}-{start + len(batch)} / {len(tickers)}")
    try:
        raw_batch = yf.download(batch, period="3mo", interval="1d", auto_adjust=False, progress=False, threads=True, group_by="ticker")
    except Exception as exc:
        print("Batch download failed:", type(exc).__name__)
        raw_batch = pd.DataFrame()
    for ticker in batch:
        item = pd.DataFrame()
        try:
            item = extract_ticker_frame(raw_batch, ticker, len(batch))
        except Exception as exc:
            print(f"YAHOO_BATCH_PARSE_WARNING {ticker} | {type(exc).__name__}")
        # Yahoo batch responses can omit individual Thai tickers even when the
        # rest of the batch succeeds. Retry only the missing ticker separately.
        if len(item) < 20:
            item = download_yahoo_history(ticker, period="3mo", attempts=2)
        if len(item) >= 20:
            short_data[ticker] = item
        else:
            missing_short.append(ticker)

print(f"Short OHLCV ready: {len(short_data)} / {len(tickers)}; unavailable/short: {len(missing_short)}")
stage_rows = []
meta_by_ticker = {}
stale_short_data = {}
MAX_STALE_DAILY_BAR_DAYS = 7
today_bkk = pd.Timestamp.now(tz="Asia/Bangkok").date()
for _, item in listed.iterrows():
    ticker = f"{item['Symbol']}.BK"
    frame = short_data.get(ticker)
    if frame is None or frame.empty:
        continue
    latest_bar_date = frame.index[-1].date()
    age_days = (today_bkk - latest_bar_date).days
    if age_days > MAX_STALE_DAILY_BAR_DAYS:
        stale_short_data[ticker] = f"Stale daily data: latest bar {latest_bar_date}; {age_days} calendar days old"
        continue
    recent = frame.tail(20)
    total20_mbaht = float((recent["Close"] * recent["Volume"]).sum() / 1_000_000.0)
    close = float(recent["Close"].iloc[-1])
    if not (MIN_PRICE_BAHT <= close <= MAX_PRICE_BAHT and total20_mbaht > MIN_TOTAL20_VALUE_MBAHT):
        continue
    stage_rows.append({"Ticker": ticker, "Symbol": item["Symbol"], "Market": item["Market"], "Sector": item["Sector"], "LastPrice": close, "Total20TradingValueMBaht": total20_mbaht})
    meta_by_ticker[ticker] = {"Market": item["Market"], "Sector": item["Sector"]}

stage = pd.DataFrame(stage_rows)
if stage.empty:
    raise RuntimeError("No stocks passed the price and total-20-session traded-value gates; stopping safely.")
print(f"Price + total 20-session value + fresh-data pass: {len(stage)}")
print(f"Excluded for stale daily data (> {MAX_STALE_DAILY_BAR_DAYS} days): {len(stale_short_data)}")

# Stage 2: slower market-cap lookup only for stocks passing the price/liquidity gates.
cap_values = []
for i, ticker in enumerate(stage["Ticker"], start=1):
    cap_baht = get_market_cap_baht(ticker)
    cap_mbaht = cap_baht / 1_000_000.0 if np.isfinite(cap_baht) else np.nan
    cap_values.append(cap_mbaht)
    if i == 1 or i % 10 == 0 or i == len(stage):
        print(f"Market cap {i}/{len(stage)} | {ticker}")
stage["MarketCapMBaht"] = cap_values
stage = stage[np.isfinite(stage["MarketCapMBaht"]) & (stage["MarketCapMBaht"] > MIN_MARKET_CAP_MBAHT)].copy()
if stage.empty:
    raise RuntimeError("No stocks passed market cap > THB 5,000m; no signals produced.")
print(f"Full universe gate passed: {len(stage)} stocks")

# Stage 3: get ~1 year of daily data for indicator warm-up and scan latest completed bar.
stock_data = {}
metadata_by_symbol = {}
scan_errors = {}
stale_indicator_data = {}
for i, row in enumerate(stage.iterrows(), start=1):
    _, record = row
    ticker = record["Ticker"]
    symbol = record["Symbol"]
    try:
        frame = download_yahoo_history(ticker, period="1y", attempts=2)
        if len(frame) < 80:
            scan_errors[ticker] = f"Insufficient completed daily bars: {len(frame)}"
            continue
        latest_bar_date = frame.index[-1].date()
        age_days = (pd.Timestamp.now(tz="Asia/Bangkok").date() - latest_bar_date).days
        if age_days > MAX_STALE_DAILY_BAR_DAYS:
            stale_indicator_data[ticker] = f"Stale indicator data: latest bar {latest_bar_date}; {age_days} calendar days old"
            continue
        stock_data[ticker] = frame
        metadata_by_symbol[ticker] = {
            "Market": record["Market"],
            "Sector": record["Sector"],
            "MarketCapMBaht": float(record["MarketCapMBaht"]),
            "Total20TradingValueMBaht": float(record["Total20TradingValueMBaht"]),
        }
    except Exception as exc:
        scan_errors[ticker] = f"{type(exc).__name__}: {exc}"
    if i == 1 or i % 10 == 0 or i == len(stage):
        print(f"Indicator history {i}/{len(stage)} | loaded={len(stock_data)} errors={len(scan_errors)}")

if not stock_data:
    raise RuntimeError("No qualifying stocks have fresh, sufficient OHLCV history. Stopping without output signals.")

# Enforce one common latest completed trading date across every published row.
expected_scan_date = max(frame.index[-1].date() for frame in stock_data.values())
for ticker in list(stock_data):
    ticker_date = stock_data[ticker].index[-1].date()
    if ticker_date != expected_scan_date:
        stale_indicator_data[ticker] = (
            f"Latest bar {ticker_date} does not match common scan date {expected_scan_date}"
        )
        stock_data.pop(ticker)
        metadata_by_symbol.pop(ticker, None)
if not stock_data:
    raise RuntimeError("No stocks share a common latest completed daily bar; stopping without publishing.")
official_set_latest_date = get_official_set_latest_trading_date()
if expected_scan_date != official_set_latest_date:
    raise RuntimeError(
        "MARKET_DATE_MISMATCH: latest completed Yahoo Finance bar "
        f"{expected_scan_date} does not match SET's official latest trading date "
        f"{official_set_latest_date}. Refusing to publish stale or incomplete scan data."
    )
print("Official market-date validation: PASS")
print(f"Common completed scan date: {expected_scan_date}")
print(f"Excluded for stale/mismatched indicator data: {len(stale_indicator_data)}")

results = scan_universe(stock_data, metadata_by_symbol, CONFIG)
if results.empty:
    raise RuntimeError("Scanner returned no rows; stopping without publishing.")
if results["Date"].isna().any() or pd.to_datetime(results["Date"], errors="coerce").dt.date.nunique() != 1:
    raise RuntimeError("REPORT_DATA_QUALITY_FAILED: scan rows do not share one valid scan date; refusing to publish.")
if (results["UniverseStatus"] == "SCAN_ERROR").any():
    scan_errors["SCANNER_INTERNAL"] = f"{int((results['UniverseStatus'] == 'SCAN_ERROR').sum())} internal scan errors"
results = results.sort_values(["EARLY_ENTRY", "ENTRY_SCORE"], ascending=[False, False], na_position="last")
signals = results[(results["EARLY_ENTRY"] == True) | (results["EARLY_WARNING"].isin(["WATCH", "WARNING"]))].copy()

print("\n" + "=" * 90)
print("LIVE SCAN COMPLETE — LATEST COMPLETED DAILY BARS ONLY")
print("No backtest | no orders | no Production changes | no GitHub Actions")
print("=" * 90)
print("Classification source:", CLASSIFICATION_SOURCE_USED)
print("Stocks passing all universe filters:", len(stage))
print("Stocks with indicator data:", len(stock_data))
print("Scan errors:", len(scan_errors))
print("Stale/mismatched data exclusions:", len(stale_short_data) + len(stale_indicator_data))
print("EARLY ENTRY:", int((results["EARLY_ENTRY"] == True).sum()))
print("WATCH/WARNING:", int(results["EARLY_WARNING"].isin(["WATCH", "WARNING"]).sum()))
display(signals if not signals.empty else results.head(20))

# Export the scan report from this Colab session (not to GitHub/main).
output_path = "/content/fast_hybrid_pro_live_scan.csv"
results.to_csv(output_path, index=False, encoding="utf-8-sig")
print("CSV report saved:", output_path)
try:
    from google.colab import files
    files.download(output_path)
except Exception:
    print("Not running in Google Colab; CSV remains at", output_path)

if scan_errors:
    print("\nFirst 20 scan errors:")
    for ticker, message in list(scan_errors.items())[:20]:
        print(f"{ticker}: {message}")


In [ ]:
# Publish this completed scan to the dedicated scan-reports branch.# Requires a Colab secret named GITHUB_TOKEN with Contents: Read and write access# to worasak2530seed/fast-hybrid-pro. No token is printed or saved in the notebook.import base64import jsonfrom datetime import datetime, timezone# Fail clearly if the scan cell above was skipped or failed; never publish partial state.if "results" not in globals() or not isinstance(globals().get("results"), pd.DataFrame):    raise RuntimeError("SCAN_NOT_RUN: run the complete scan cell immediately above and fix any error before publishing.")if "signals" not in globals() or not isinstance(globals().get("signals"), pd.DataFrame):    raise RuntimeError("SCAN_NOT_RUN: scan output 'signals' is missing; rerun the scan cell before publishing.")REPORT_BRANCH = "scan-reports"REPORT_REPO = "worasak2530seed/fast-hybrid-pro"REPORT_API = f"https://api.github.com/repos/{REPORT_REPO}/contents"RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")UPDATED_AT = datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")SCAN_DATE = str(pd.to_datetime(results["Date"], errors="coerce").max().date()) if "Date" in results and results["Date"].notna().any() else "unknown"GITHUB_TOKEN = Nonetry:    from google.colab import userdata    for secret_name in ("FAST_HYBRID_GITHUB_TOKEN", "GITHUB_TOKEN", "GH_TOKEN"):        try:            GITHUB_TOKEN = userdata.get(secret_name)        except Exception:            GITHUB_TOKEN = None        if GITHUB_TOKEN:            breakexcept Exception:    GITHUB_TOKEN = Noneif not GITHUB_TOKEN:    # Fallback for Colab sessions where Secrets are unavailable/not authorized.    # The token is entered interactively, hidden by getpass, and is not written to the notebook.    from getpass import getpass    GITHUB_TOKEN = getpass("GitHub token (input hidden; never paste it into a code cell): ").strip()if not GITHUB_TOKEN:    raise RuntimeError(        "GITHUB_REPORT_PUBLISH_FAILED: no GitHub token was provided. "        "Add FAST_HYBRID_GITHUB_TOKEN (preferred) or GITHUB_TOKEN in Colab Secrets, or rerun this cell and enter the token at the hidden prompt. "        "The local CSV was already created; no Telegram success notification should be sent."    )REPORT_HEADERS = {    "Authorization": f"Bearer {GITHUB_TOKEN}",    "Accept": "application/vnd.github+json",    "X-GitHub-Api-Version": "2022-11-28",    "User-Agent": "Fast-Hybrid-Pro-Scanner",}# Verify authentication and branch access before attempting any report writes.auth_check = requests.get(    f"https://api.github.com/repos/{REPORT_REPO}/branches/{REPORT_BRANCH}",    headers=REPORT_HEADERS,    timeout=30,)if auth_check.status_code in (401, 403):    raise RuntimeError(        "GITHUB_REPORT_PUBLISH_FAILED: GitHub rejected the token or it lacks repository access. "        "Confirm the token belongs to the correct GitHub account and has Contents: Read and write "        "for worasak2530seed/fast-hybrid-pro. Do not paste the token into chat."    )if auth_check.status_code == 404:    raise RuntimeError(        "GITHUB_REPORT_PUBLISH_FAILED: branch scan-reports was not found or is not visible to this token."    )auth_check.raise_for_status()print("GITHUB_TOKEN_AUTH_CHECK_PASS")def publish_report_file(path, raw_bytes, message):    url = f"{REPORT_API}/{path}"    existing = requests.get(url, headers=REPORT_HEADERS, params={"ref": REPORT_BRANCH}, timeout=30)    payload = {        "message": message,        "content": base64.b64encode(raw_bytes).decode("ascii"),        "branch": REPORT_BRANCH,    }    if existing.status_code == 200:        payload["sha"] = existing.json()["sha"]    elif existing.status_code != 404:        existing.raise_for_status()    response = requests.put(url, headers=REPORT_HEADERS, json=payload, timeout=45)    response.raise_for_status()    return response.json()["content"]["html_url"]# Upload data first; publish SUCCESS status last so Termux cannot announce success early.csv_bytes = results.to_csv(index=False).encode("utf-8-sig")summary_rows = []for _, row in signals.iterrows():    summary_rows.append({        "Symbol": str(row.get("Symbol", "")),        "Date": str(row.get("Date", "")),        "Close": None if pd.isna(row.get("Close")) else float(row.get("Close")),        "EARLY_ENTRY": bool(row.get("EARLY_ENTRY", False)),        "ENTRY_SCORE": None if pd.isna(row.get("ENTRY_SCORE")) else int(row.get("ENTRY_SCORE")),        "EARLY_WARNING": str(row.get("EARLY_WARNING", "")),        "SignalReasons": str(row.get("SignalReasons", "")),        "ReferenceTarget7Pct": None if pd.isna(row.get("ReferenceTarget7Pct")) else float(row.get("ReferenceTarget7Pct")),        "ReferenceTarget15Pct": None if pd.isna(row.get("ReferenceTarget15Pct")) else float(row.get("ReferenceTarget15Pct")),    })summary = {    "run_id": RUN_ID,    "status": "SUCCESS",    "scan_date": SCAN_DATE,    "updated_at_utc": UPDATED_AT,    "universe_pass_count": int(len(stage)),    "indicator_data_count": int(len(stock_data)),    "scan_error_count": int(len(scan_errors)),    "stale_data_excluded_count": int(len(stale_short_data) + len(stale_indicator_data)),    "stale_data_exclusions": {**stale_short_data, **stale_indicator_data},    "early_entry_count": int((results["EARLY_ENTRY"] == True).sum()),    "watch_warning_count": int(results["EARLY_WARNING"].isin(["WATCH", "WARNING"]).sum()),    "signal_count": int(len(signals)),    "signals": summary_rows,    "report_path": "reports/latest_scan.csv",    "summary_path": "reports/latest_scan_summary.json",    "data_note": "20-session traded value is estimated as Close × Volume; it is not the exchange's exact traded-value field.",}summary_bytes = json.dumps(summary, ensure_ascii=False, indent=2).encode("utf-8")publish_status = "SUCCESS" if not scan_errors else "PARTIAL"status = {    "run_id": RUN_ID,    "status": publish_status,    "detail": (        "Live scanner completed and reports published."        if not scan_errors else        f"Live scanner report published with {len(scan_errors)} scan error(s); review summary before relying on results."    ),    "scan_date": SCAN_DATE,    "updated_at_utc": UPDATED_AT,    "report_path": "reports/latest_scan.csv",    "summary_path": "reports/latest_scan_summary.json",    "summary_url": f"https://github.com/{REPORT_REPO}/blob/{REPORT_BRANCH}/reports/latest_scan_summary.json",    "report_url": f"https://github.com/{REPORT_REPO}/blob/{REPORT_BRANCH}/reports/latest_scan.csv",}status_bytes = json.dumps(status, ensure_ascii=False, indent=2).encode("utf-8")csv_url = publish_report_file("reports/latest_scan.csv", csv_bytes, f"Publish scanner CSV report {RUN_ID}")summary_url = publish_report_file("reports/latest_scan_summary.json", summary_bytes, f"Publish scanner summary {RUN_ID}")status_url = publish_report_file("reports/latest_scan_status.json", status_bytes, f"Publish scanner success status {RUN_ID}")print("GITHUB_REPORT_PUBLISH_PASS")print("PUBLISH_STATUS:", publish_status)print("RUN_ID:", RUN_ID)print("SCAN_DATE:", SCAN_DATE)print("EARLY_ENTRY:", summary["early_entry_count"])print("WATCH_WARNING:", summary["watch_warning_count"])print("SCAN_ERRORS:", summary["scan_error_count"])print("STALE_DATA_EXCLUDED:", summary["stale_data_excluded_count"])print("CSV_REPORT:", csv_url)print("SUMMARY_REPORT:", summary_url)print("STATUS_REPORT:", status_url)